In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.6 Magnetostatics and the Vector Potential

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.6",
    title="Magnetostatics and the Vector Potential",
    blurb="Magnetism from steady currents: the Biot–Savart law, Ampère's law, the "
    "absence of magnetic monopoles (∇·B = 0), and the vector potential A — whose "
    "freedom to be redefined is our first encounter with gauge invariance.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

Everything in this notebook is the magnetic mirror of the three that came before.
Where static charges made an electric field, **steady currents** make a magnetic
field; where Coulomb's law summed contributions from charge elements, the
**Biot–Savart law** sums them from current elements; where Gauss's law turned
symmetry into the field, **Ampère's law** does the same for magnetism. We will draw
those parallels at every step, because the mirror is the fastest way to learn
magnetostatics once electrostatics is in hand.

Two things, though, are genuinely new. First, the magnetic field **circulates
around** its source rather than pointing toward it, a fact encoded by the **cross
product** in Biot–Savart, which we introduce in line with the physics. And there is
no magnetic charge: $\nabla\cdot\mathbf B=0$ everywhere, the second of the four field
equations the volume assembles, the magnetic counterpart of
$\nabla\cdot\mathbf E=\rho/\varepsilon_0$ but with zero on the right.

Because $\mathbf B$ is divergence-free, it can always be written as the curl of a
**vector potential**, $\mathbf B=\nabla\times\mathbf A$, the magnetic analogue of
the scalar potential $V$ but a vector. And here we meet an idea that will grow to
dominate the rest of physics: $\mathbf A$ is **not unique**. We can add the gradient
of any scalar to it without changing $\mathbf B$ at all. That freedom is **gauge
invariance**. We meet it concretely and computationally here, show two different
$\mathbf A$ fields giving the identical $\mathbf B$, and plant the arc that runs
forward to Maxwell ([§3.8](maxwell-waves.ipynb)), the relativistic field tensor
([§3.12](relativistic-maxwell.ipynb)), and the quantum
Aharonov–Bohm effect (Vol VI).

Everything is in **SI units**, with $\mu_0 = 4\pi\times10^{-7}\,$T·m/A. Steady
currents make static fields, so every figure here is a still; there is no motion to
animate.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a Biot–Savart integral matching $\mu_0 I/2\pi r$, a circulation
> equal to $\mu_0 I$, a numerical curl recovering $\mathbf B$, two gauges differing
> by an exact gradient. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.

> **Scope.** A working review, not a full course. See Nolting, *Theoretical Physics
> 3* {cite}`nolting3`; Griffiths, *Introduction to Electrodynamics*
> {cite}`griffiths_em` (ch. 5); Jackson {cite}`jackson` (ch. 5).

## Theory in brief

### Steady currents and the magnetic field

Magnetostatics treats steady currents (constant in time, $\partial_t\rho=0$, so
$\nabla\cdot\mathbf J=0$) and the static magnetic field they produce. It runs
parallel to electrostatics throughout, with current density $\mathbf J$ in the role
of charge density and $\mathbf B$ in the role of $\mathbf E$.

### The Biot–Savart law

The field of a current element $I\,d\boldsymbol\ell$ is

```{math}
:label: eq-biot-savart
d\mathbf B = \frac{\mu_0}{4\pi}\,\frac{I\,d\boldsymbol\ell\times\hat{\mathbf r}}{r^2},
```

the magnetic mirror of Coulomb's $d\mathbf E=(1/4\pi\varepsilon_0)\,dq\,
\hat{\mathbf r}/r^2$. The decisive difference is the **cross product**: the field is
perpendicular to both the current and the line of sight, so $\mathbf B$
*circulates around* the current (right-hand rule) instead of pointing toward it.

### Ampère's law

The magnetic analogue of Gauss's law is **Ampère's law**,

```{math}
:label: eq-ampere
\oint \mathbf B\cdot d\boldsymbol\ell = \mu_0\,I_{\text{enc}},
```

the line integral of $\mathbf B$ around a closed loop equals $\mu_0$ times the
current threading it. When symmetry makes $\mathbf B$ constant and tangent along a
well-chosen loop, it gives the field directly (wire, solenoid, toroid). Its
differential form is $\nabla\times\mathbf B=\mu_0\mathbf J$. Both forms follow from
the Biot–Savart law by a curl computation that Griffiths {cite}`griffiths_em`
(ch. 5) carries out in full.

### No magnetic monopoles

There is no magnetic charge, so

```{math}
:label: eq-divB
\nabla\cdot\mathbf B = 0 \quad\text{everywhere, always.}
```

Field lines of $\mathbf B$ never begin or end; they close on themselves. This is the
second of the four static field equations, the magnetic counterpart of
$\nabla\cdot\mathbf E=\rho/\varepsilon_0$ with zero on the right.

### The vector potential

Any divergence-free field is a curl (a consequence of the Helmholtz theorem;
Griffiths {cite}`griffiths_em`, appendix B, proves it), so {eq}`eq-divB` lets us write

```{math}
:label: eq-vector-potential
\mathbf B = \nabla\times\mathbf A,
```

with $\mathbf A$ the **vector potential**, the magnetic analogue of $V$ but a
vector. Writing $\mathbf B$ this way builds $\nabla\cdot\mathbf B=0$ in automatically
(the divergence of a curl is identically zero).

### Gauge freedom

The vector potential is **not unique**. Because the curl of a gradient vanishes,
$\nabla\times(\nabla\chi)=0$, the transformation

```{math}
:label: eq-gauge
\mathbf A \to \mathbf A + \nabla\chi
```

leaves $\mathbf B=\nabla\times\mathbf A$ unchanged for *any* scalar field $\chi$.
This freedom to redefine $\mathbf A$ without touching the physics is **gauge
invariance**. A common way to use up the freedom is the **Coulomb gauge**
$\nabla\cdot\mathbf A=0$, a convenient *choice*, not a physical constraint. We make
all of this concrete in Exercise 7, and pitch the larger arc there.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import draw, validate

MU0 = 4.0e-7 * np.pi  # vacuum permeability, T·m/A
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT


def biot_savart(path, current, P):
    """Biot-Savart field of a current path (eq-biot-savart).

    Sums (μ0·I/4π)·dl × R̂/R^2 (`numpy.cross`) over the path's straight
    segments; a segment loop keeps memory O(P) even on a 3-D grid.

    Parameters
    ----------
    path : numpy.ndarray
        Points ``(M, 3)`` tracing the wire.
    current : float
        Current $I$ in amperes.
    P : numpy.ndarray
        Field points of shape ``(..., 3)``.

    Returns
    -------
    numpy.ndarray
        The magnetic field at ``P``, in tesla.
    """
    path = np.asarray(path, float)
    P = np.asarray(P, float)
    mids = 0.5 * (path[1:] + path[:-1])
    dls = path[1:] - path[:-1]
    B = np.zeros_like(P)
    for dl, mid in zip(dls, mids):
        R = P - mid
        Rn = np.linalg.norm(R, axis=-1)
        with np.errstate(divide="ignore", invalid="ignore"):
            B = B + np.cross(dl, R) / Rn[..., None] ** 3
    return MU0 * current / (4.0 * np.pi) * np.nan_to_num(B)


def curl_3d(Fx, Fy, Fz, x, y, z):
    """Numerical curl of a vector field on a 3-D grid.

    Central differences (`numpy.gradient`) on an ``indexing='ij'`` grid; the
    discrete form of the curl that detects circulation.

    Parameters
    ----------
    Fx, Fy, Fz : numpy.ndarray
        The field components on the grid.
    x, y, z : numpy.ndarray
        1-D coordinate arrays for the three axes.

    Returns
    -------
    tuple of numpy.ndarray
        The three components of the curl.
    """
    dFz_dy = np.gradient(Fz, y, axis=1)
    dFy_dz = np.gradient(Fy, z, axis=2)
    dFx_dz = np.gradient(Fx, z, axis=2)
    dFz_dx = np.gradient(Fz, x, axis=0)
    dFy_dx = np.gradient(Fy, x, axis=0)
    dFx_dy = np.gradient(Fx, y, axis=1)
    return dFz_dy - dFy_dz, dFx_dz - dFz_dx, dFy_dx - dFx_dy


def div_3d(Fx, Fy, Fz, x, y, z):
    """Numerical divergence of a vector field on a 3-D grid.

    Central differences (`numpy.gradient`) on an ``indexing='ij'`` grid;
    should vanish for B away from currents.

    Parameters
    ----------
    Fx, Fy, Fz : numpy.ndarray
        The field components.
    x, y, z : numpy.ndarray
        1-D coordinate arrays.

    Returns
    -------
    numpy.ndarray
        The divergence on the grid.
    """
    return (
        np.gradient(Fx, x, axis=0)
        + np.gradient(Fy, y, axis=1)
        + np.gradient(Fz, z, axis=2)
    )


def straight_wire(length=200.0, n=4000):
    """A long straight wire along the z-axis (approximately infinite).

    A path from −L/2 to L/2 to feed :func:`biot_savart`.

    Parameters
    ----------
    length : float, optional
        Total wire length (default 200.0).
    n : int, optional
        Number of points (default 4000).

    Returns
    -------
    numpy.ndarray
        The wire path of shape ``(n, 3)``.
    """
    z = np.linspace(-length / 2, length / 2, n)
    return np.stack([np.zeros_like(z), np.zeros_like(z), z], axis=-1)


def current_loop(radius, n=400, z0=0.0):
    """A circular current loop in a plane of constant z.

    A closed path tracing a ring of given radius, for :func:`biot_savart`.

    Parameters
    ----------
    radius : float
        Loop radius.
    n : int, optional
        Number of points (default 400).
    z0 : float, optional
        Plane height (default 0.0).

    Returns
    -------
    numpy.ndarray
        The loop path of shape ``(n, 3)``.
    """
    phi = np.linspace(0.0, 2.0 * np.pi, n)
    return np.stack(
        [radius * np.cos(phi), radius * np.sin(phi), np.full_like(phi, z0)], axis=-1
    )

## Exercise 1 — Biot–Savart: the straight wire

The field of a current element {eq}`eq-biot-savart` is built from a **cross product**,
$d\mathbf B\propto I\,d\boldsymbol\ell\times\hat{\mathbf r}$, which is what makes the
magnetic field circulate *around* the current rather than point toward it like
Coulomb's. Summed along a long straight wire on the $z$-axis, the contributions give
a purely azimuthal field whose magnitude Ampère's law (Exercise 4) will confirm is
$\mu_0 I/2\pi r$ ({numref}`fig-ms-wire-setup`).

**This exercise.** Take $I=1\,$A on a wire from $z=-100$ to $z=+100\,$m
(4000 segments, effectively infinite).

1. Integrate the Biot–Savart law numerically (a segment sum with
   `numpy.cross`, in the `biot_savart` helper) at field points $r=0.1$ and
   $0.2\,$m in the $z=0$ plane, and compare $|\mathbf B|$ to
   $\mu_0 I/2\pi r$.
2. Confirm the field is azimuthal (perpendicular to both the wire and the
   radius) ({numref}`fig-ms-wire-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    ratio_wire, 1.0, "Biot–Savart reproduces the wire field μ₀I/2πr", rtol=1e-3
)
validate.close(
    azimuthal,
    0.0,
    "the wire's field is azimuthal (perpendicular to wire and radius)",
    atol=1e-6,
)

## Exercise 2 — Biot–Savart: the circular loop

A current loop is the magnetic workhorse: it is the elementary magnetic dipole, and
its on-axis field has a clean closed form. Summing Biot–Savart around the ring, the
transverse contributions cancel by symmetry and only the axial component survives
({numref}`fig-ms-loop-setup`),

$$ B_z(z) = \frac{\mu_0 I R^2}{2\,(R^2+z^2)^{3/2}}. $$

**This exercise.** Take a loop of radius $R=0.05\,$m carrying $I=1\,$A in
the $xy$-plane (400 segments).

1. Integrate the Biot–Savart law (the `biot_savart` helper, `numpy.cross`
   summed over the ring) for the on-axis $B_z$ at $z=0,\,0.05,\,0.1\,$m and
   compare to the closed form.
2. Map the field in the axial plane ({numref}`fig-ms-loop-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    max_loop_err,
    0.0,
    "the loop's on-axis field matches the closed form μ₀IR²/2(R²+z²)^{3/2}",
    atol=1e-3,
)

## Exercise 3 — No magnetic monopoles: ∇·B = 0

The deepest structural fact of magnetostatics is that there is no magnetic charge:
$\nabla\cdot\mathbf B=0$ everywhere {eq}`eq-divB`. Field lines of $\mathbf B$ never
start or stop. We test it on the loop's field by taking the **numerical divergence**
(a sum of central differences with `numpy.gradient`, the `div_3d` helper) on a
3-D grid, away from the ring itself.

**This exercise.** Build the loop's $\mathbf B$ (Exercise 2) on a $31^3$ grid
over $[-0.15, 0.15]^3\,$m, compute $\nabla\cdot\mathbf B$ with the `div_3d`
helper, and confirm it vanishes in the region away from the wire. The exact value is identically
zero; the small residual that remains is pure discretization (the central
differences straddle the steep field near the ring), so we measure it on nodes a few
grid steps from the current and scale the tolerance to the grid, the same philosophy
as the corner discontinuity of [§3.4](laplace-poisson.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    divB_rel < 1e-2,
    "∇·B vanishes away from the current — there are no magnetic monopoles",
    f"relative residual {divB_rel:.2e} (pure discretization; → 0 as the grid refines)",
)

## Exercise 4 — Ampère's law

Ampère's law {eq}`eq-ampere` is the magnetic Gauss's law: $\oint\mathbf B\cdot
d\boldsymbol\ell=\mu_0 I_{\text{enc}}$, the circulation of $\mathbf B$ around any
closed loop equals $\mu_0$ times the enclosed current, independent of the loop's
size or shape ({numref}`fig-ms-ampere-setup`). For the straight wire it turns the
azimuthal symmetry directly into $B=\mu_0 I/2\pi r$.

**This exercise.** For the wire of Exercise 1 ($I=1\,$A), compute the
circulation $\oint\mathbf B\cdot d\boldsymbol\ell$ around a circular Amperian
loop of radius $d=0.15\,$m in the $z=0$ plane (the field from `biot_savart`,
the line integral by `numpy.trapezoid` over the azimuth), and confirm it
equals $\mu_0 I_{\text{enc}}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(circulation, MU0 * I_wire, "Ampère's law: ∮B·dl = μ₀I_enc", rtol=1e-3)

## Exercise 5 — The solenoid

A solenoid is Ampère's law's showcase: a long, tightly wound coil with $n$ turns per
unit length carrying current $I$. A rectangular Amperian loop with one side inside
and one far outside ({numref}`fig-ms-solenoid-setup`) gives a **uniform** interior
field $B=\mu_0 n I$ along the axis and (ideally) zero outside. We confirm it by
superposing the Biot–Savart fields of many individual loops.

**This exercise (student).** Take $n=1000$ turns/m and $I=1\,$A. From
Ampère's law the interior field is $\mu_0 n I$.

1. Confirm it by building a finite solenoid as a stack of current loops
   (radius $0.02\,$m, length $0.4\,$m, summed with `biot_savart`),
   evaluating $B_z$ on the axis at the centre, and comparing to $\mu_0 n I$
   (a finite-length tolerance applies).
2. Check the field is far smaller outside
   ({numref}`fig-ms-solenoid-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(Bz_center, B_ideal, "the solenoid interior field is μ₀nI", rtol=2e-2)
validate.check(
    abs(Bz_outside) < 0.1 * abs(Bz_center),
    "the solenoid field is far weaker outside than inside",
    f"outside/inside = {abs(Bz_outside / Bz_center):.3f}",
)

## Exercise 6 — The vector potential: B = ∇×A

Because $\nabla\cdot\mathbf B=0$ {eq}`eq-divB`, the field is always a curl,
$\mathbf B=\nabla\times\mathbf A$ {eq}`eq-vector-potential`. To work with $\mathbf A$
we need the **numerical curl**, $\nabla\times\mathbf A$, built from central
differences of the components (the `curl_3d` helper, `numpy.gradient`). We test
it on a case with a known answer: a uniform field $\mathbf B=B_0\hat{\mathbf z}$,
whose symmetric ("Coulomb-gauge") vector potential is
$\mathbf A=\tfrac12\mathbf B\times\mathbf r=\tfrac{B_0}{2}(-y,\,x,\,0)$.

**This exercise.** On a $21^3$ grid over $[-1,1]^3$, build
$\mathbf A=\tfrac{B_0}{2}(-y,x,0)$ with $B_0=2\,$T, take its numerical curl
with the `curl_3d` helper, and confirm it recovers the uniform
$\mathbf B=B_0\hat{\mathbf z}$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    curl_err,
    0.0,
    "B = ∇×A recovers the uniform field B₀ẑ from the vector potential",
    atol=1e-4,
)

## Exercise 7 — Gauge freedom

Here is the idea that grows to organize all of physics, made concrete. The vector
potential is **not unique**: adding $\nabla\chi$ for any scalar $\chi$ leaves
$\mathbf B=\nabla\times\mathbf A$ unchanged {eq}`eq-gauge`, because
$\nabla\times(\nabla\chi)=0$. We demonstrate it with two genuinely different
potentials for the *same* uniform $\mathbf B=B_0\hat{\mathbf z}$: the symmetric
Coulomb-gauge $\mathbf A_1=\tfrac{B_0}{2}(-y,x,0)$ and the Landau-gauge
$\mathbf A_2=(-B_0 y,0,0)$. Both have the same curl, they differ by an exact
gradient, and $\mathbf A_1$ satisfies the convenient **Coulomb gauge**
$\nabla\cdot\mathbf A=0$.

**This exercise.** On the $21^3$ grid, take
$\mathbf A_1=\tfrac{B_0}{2}(-y,x,0)$ and $\mathbf A_2=(-B_0 y,0,0)$.

1. Confirm with the `curl_3d` helper that
   $\nabla\times\mathbf A_1=\nabla\times\mathbf A_2=B_0\hat{\mathbf z}$.
2. Confirm $\mathbf A_1-\mathbf A_2$ equals $\nabla\chi$ for
   $\chi=B_0 xy/2$ (an exact gauge transformation).
3. Verify $\mathbf A_1$ satisfies $\nabla\cdot\mathbf A=0$ with the
   `div_3d` helper.

This freedom is **gauge invariance**, and it is no accident of magnetostatics. It
returns as a *tool* in [§3.8](maxwell-waves.ipynb), where the Lorenz gauge decouples
Maxwell's equations into
wave equations; as *structure* in the relativistic capstone
[§3.12](relativistic-maxwell.ipynb), where the
four-potential and the manifestly gauge-invariant field tensor make it geometric;
and as *physics* in the quantum volume (Vol VI), where the Aharonov–Bohm effect shows
$\mathbf A$ has observable consequences even where $\mathbf B=0$. Its root is the
symmetry–conservation link of Noether
([§2.2](../02-classical-mechanics/noether.ipynb)): a global phase symmetry gives charge
conservation, and local gauge symmetry is its generalization. We plant the arc here
and move on.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    gauge_same,
    0.0,
    "two gauges (Coulomb and Landau) give the same physical field B",
    atol=1e-4,
)
validate.close(
    grad_err,
    0.0,
    "the two vector potentials differ by a pure gradient ∇χ — a gauge transformation",
    atol=1e-4,
)
validate.close(
    divA1_max,
    0.0,
    "the symmetric choice satisfies the Coulomb gauge ∇·A = 0",
    atol=1e-6,
)

## Exercise 8 — The magnetic dipole

Far from a current loop, the field takes the **magnetic dipole** form, the exact
mirror of the electric dipole of
[§3.1](coulomb-field.ipynb)/[§3.5](multipole-expansion.ipynb): it falls as $1/r^3$
and is set by the
magnetic moment $\mathbf m=I\pi R^2\hat{\mathbf z}$
({numref}`fig-ms-dipole-setup`). On axis, the loop's field
$\mu_0 I R^2/2(R^2+z^2)^{3/2}$ reduces for $z\gg R$ to $\mu_0 m/2\pi z^3$, the $1/r^3$
dipole falloff. There is no magnetic monopole term (the series starts at the dipole),
because $\nabla\cdot\mathbf B=0$.

**This exercise (student).** For the loop of Exercise 2 ($R=0.05\,$m,
$I=1\,$A):

1. Evaluate the on-axis $B_z$ with `biot_savart` at $z$ and $2z$ in the far
   zone ($z=0.5\,$m) and confirm the ratio is $\approx 8$, the signature of
   a $1/r^3$ dipole.
2. Compare the far field to $\mu_0 m/2\pi z^3$ with $m=I\pi R^2$
   ({numref}`fig-ms-dipole-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    falloff_ratio,
    8.0,
    "the current loop's far field falls as 1/r³ (a magnetic dipole)",
    rtol=2e-2,
)

## Exercise 9 — The four field equations so far, and what is missing

Step back and collect what we have built. Across electrostatics and magnetostatics
we have assembled four field equations:

$$ \nabla\cdot\mathbf E = \frac{\rho}{\varepsilon_0}\ (3.3), \qquad
\nabla\cdot\mathbf B = 0\ (\text{here}), \qquad
\nabla\times\mathbf E = 0\ (3.2), \qquad
\nabla\times\mathbf B = \mu_0\mathbf J\ (\text{here}). $$

These are **Maxwell's equations in the static limit**. They are almost the whole
story, but two of them change the moment fields vary in time. A changing magnetic
field makes $\nabla\times\mathbf E$ pick up a term $-\partial_t\mathbf B$ (Faraday's
induction, [§3.7](induction.ipynb)), so the electric field is no longer curl-free. And
consistency with charge conservation forces $\nabla\times\mathbf B$ to gain a term
$\mu_0\varepsilon_0\,\partial_t\mathbf E$ (Maxwell's displacement current,
[§3.8](maxwell-waves.ipynb)). With those two additions the equations couple $\mathbf E$ and $\mathbf B$ into
a self-sustaining wave, and **light appears**.

**This exercise.** Close the static story with a *measurement* of the fourth
equation in its current-free case: away from the ring the current density is
zero, so Ampère's differential law demands $\nabla\times\mathbf B=0$ there.
Compute the numerical curl of the loop field already gridded in Exercise 3
(the `curl_3d` helper) and confirm it vanishes off the current, just as the
divergence did. The time-derivative terms are identically zero here (the
field is static), so neither dynamic Maxwell term is active yet. We are two
terms, and two notebooks, from light.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    curlB_rel < 5e-2,
    "∇×B vanishes off the current (Ampère's differential law, measured), closing the static quartet",
    f"relative curl residual {curlB_rel:.2e} away from the ring",
)

## Notebook summary

- A vectorised Biot–Savart integrator (`numpy.cross`): the straight wire $B=\mu_0I/2\pi r$
  (azimuthal) and the loop on-axis $\mu_0IR^2/2(R^2+z^2)^{3/2}$.
- $\nabla\cdot\mathbf B=0$ away from the current (no monopoles); Ampère's $\oint\mathbf B
  \cdot d\boldsymbol\ell=\mu_0 I$ (`numpy.trapezoid`); the solenoid interior $\mu_0 nI$;
  the vector potential $\mathbf B=\nabla\times\mathbf A$ by numerical curl; and the loop's
  $1/r^3$ magnetic-dipole far field.
- The gauge-freedom centrepiece: Coulomb- and Landau-gauge $\mathbf A$ giving the same
  $\mathbf B$ and differing by an exact gradient $\nabla\chi$. The four static field
  equations assembled, with the two time-derivative terms
  ([§3.7](induction.ipynb), [§3.8](maxwell-waves.ipynb)) still zero.

## Outlook

- **The magnetic scalar potential.** In current-free regions $\nabla\times\mathbf
  B=0$, so $\mathbf B$ can briefly be written as $-\nabla\Omega$ like an electric
  field; it fails wherever current flows, which is why the *vector* potential is the
  general tool.
- **Forces and energy.** The force between currents defines the ampere; a current
  distribution stores magnetic energy $\tfrac12 L I^2$, the magnetic mirror of the
  capacitor's $\tfrac12 C V^2$.
- **Gauges and their uses.** The Coulomb gauge here versus the Lorenz gauge that
  decouples Maxwell's equations ([§3.8](maxwell-waves.ipynb)); the Aharonov–Bohm
  effect, where $\mathbf A$ is
  physically observable even where $\mathbf B=0$ (Vol VI).
- **Forward links.** Induction ([§3.7](induction.ipynb)), the displacement current
  and electromagnetic
  waves ([§3.8](maxwell-waves.ipynb)), the four-potential and field tensor
  ([§3.12](relativistic-maxwell.ipynb)), gauge invariance in
  quantum mechanics (Vol VI), and its Noether root
  ([§2.2](../02-classical-mechanics/noether.ipynb)).

## References

```{bibliography}
:filter: docname in docnames
```